In [276]:
import requests

In [277]:
URL:str = "https://solved.ac/api/v3/problem/lookup"

In [278]:
NUM_FROM:int = 1000
NUM_TO:int   = 1099

In [279]:
def getParams (num_from:int, num_to:int) -> str:
    result:str = ""
    for _ in range(num_from, num_to + 1, 1):
        result += f"{_},"
    return result[:-1]

In [280]:
def getRespone (url:str, params:str) -> requests.Response:
    return requests.get(url=url, params={"problemIds": params})

In [281]:
def responseBodyToJson (response:requests.Response) -> any:
    return response.json()

In [282]:
response=getRespone(url=URL, params=getParams(num_from=NUM_FROM, num_to=NUM_TO))

response

<Response [200]>

In [283]:
responseBody = responseBodyToJson(response)

len(responseBody)

100

In [284]:
responseBodyDict:dict = {}

for _ in responseBody:
    responseBodyDict[_["problemId"]] = _

len(responseBodyDict)

100

In [285]:
df_columns:list = list(responseBodyDict[list(responseBodyDict.keys())[0]].keys())
df_columns

['problemId',
 'titleKo',
 'titles',
 'isSolvable',
 'isPartial',
 'acceptedUserCount',
 'level',
 'votedUserCount',
 'sprout',
 'givesNoRating',
 'isLevelLocked',
 'averageTries',
 'official',
 'tags',
 'metadata']

In [286]:
import pandas as pd

In [287]:
df_data = []

for _ in responseBodyDict.keys():
    df_data_row:list = []
    for data in df_columns:
        df_data_row.append(responseBodyDict[_][data])
    df_data.append(df_data_row)

len(df_data)

100

In [288]:
df = pd.DataFrame(df_data, columns=df_columns)
df.index = df["problemId"]
del df["problemId"]

In [289]:
df

,titleKo,titles,isSolvable,isPartial,acceptedUserCount,level,votedUserCount,sprout,givesNoRating,isLevelLocked,averageTries,official,tags,metadata
problemId,,,,,,,,,,,,,,
1000,A+B,"[{'language': 'en', 'languageDisplayName': 'en...",True,False,300611,1,292,True,False,True,2.5655,True,"[{'key': 'implementation', 'isMeta': False, 'b...",{}
1001,A-B,"[{'language': 'ko', 'languageDisplayName': 'ko...",True,False,257547,1,113,True,False,True,1.4399,True,"[{'key': 'implementation', 'isMeta': False, 'b...",{}
1002,터렛,"[{'language': 'ko', 'languageDisplayName': 'ko...",True,False,37787,8,285,False,False,False,4.4299,True,"[{'key': 'case_work', 'isMeta': False, 'bojTag...",{}
1003,피보나치 함수,"[{'language': 'ko', 'languageDisplayName': 'ko...",True,False,54915,8,250,False,False,False,2.9960,True,"[{'key': 'dp', 'isMeta': False, 'bojTagId': 25...",{}
1004,어린 왕자,"[{'language': 'ko', 'languageDisplayName': 'ko...",True,False,15979,8,165,False,False,False,2.1610,True,"[{'key': 'geometry', 'isMeta': False, 'bojTagI...",{}
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1095,마법의 구슬,"[{'language': 'ko', 'languageDisplayName': 'ko...",True,False,187,16,36,False,False,False,5.5561,True,"[{'key': 'math', 'isMeta': False, 'bojTagId': ...",{}
1096,종이 접기,"[{'language': 'ko', 'languageDisplayName': 'ko...",True,False,75,18,13,False,False,False,3.2133,True,"[{'key': 'bruteforcing', 'isMeta': False, 'boj...",{}
1097,마법의 문자열,"[{'language': 'ko', 'languageDisplayName': 'ko...",True,False,362,16,48,False,False,False,2.0331,True,"[{'key': 'bruteforcing', 'isMeta': False, 'boj...",{}


In [290]:
import os
import datetime

In [291]:
DATA_DIR:str = "./data"

DATA_DIR_RAW:str = DATA_DIR + "/raw"
DATA_DIR_MODIFIED:str = DATA_DIR + "/modified"

In [292]:
if not os.path.isdir(DATA_DIR):
    os.makedirs(DATA_DIR)
    
if not os.path.isdir(DATA_DIR_RAW):
    os.makedirs(DATA_DIR_RAW)

if not os.path.isdir(DATA_DIR_MODIFIED):
    os.makedirs(DATA_DIR_MODIFIED)

In [293]:
date_str = datetime.datetime.now().strftime("%Y%m%d%H%M%S")
file:str = f"{DATA_DIR_RAW}/boj.problem.raw.{NUM_FROM}.{NUM_TO}.{date_str}.csv"

In [294]:
df.to_csv(file)

In [295]:
df_copy = df.copy()

In [296]:
title_en:list[str] = []

for i in list(df_copy.index):
    temp = ""
    for title in df_copy.loc[i]["titles"]:
        if title["language"] == "en":
            temp = title["title"]
            break
    title_en.append(temp)

len(title_en)

100

In [297]:
tags:list[list[dict]] = []

for i in list(df_copy.index):
    tag:list[dict] = []
    for tagline in df_copy.loc[i]["tags"]:
        temp:dict = {
            "tagBOJ": tagline["bojTagId"],
            "tagSA" : tagline["key"]
        }
        tag.append(temp)
    tags.append(tag)

len(tags)

100

In [298]:
df_copy = df_copy.rename(columns={"titles": "titleEn"})
del df_copy["isSolvable"]
del df_copy["votedUserCount"]
del df_copy["givesNoRating"]
del df_copy["isLevelLocked"]
del df_copy["official"]
del df_copy["metadata"]
del df_copy["isPartial"]

In [299]:
df_copy["titleEn"] = title_en
df_copy["tags"] = tags

In [300]:
df_copy

,titleKo,titleEn,acceptedUserCount,level,sprout,averageTries,tags
problemId,,,,,,,
1000,A+B,A+B,300611,1,True,2.5655,"[{'tagBOJ': 102, 'tagSA': 'implementation'}, {..."
1001,A-B,,257547,1,True,1.4399,"[{'tagBOJ': 102, 'tagSA': 'implementation'}, {..."
1002,터렛,,37787,8,False,4.4299,"[{'tagBOJ': 137, 'tagSA': 'case_work'}, {'tagB..."
1003,피보나치 함수,,54915,8,False,2.9960,"[{'tagBOJ': 25, 'tagSA': 'dp'}]"
1004,어린 왕자,,15979,8,False,2.1610,"[{'tagBOJ': 100, 'tagSA': 'geometry'}, {'tagBO..."
...,...,...,...,...,...,...,...
1095,마법의 구슬,,187,16,False,5.5561,"[{'tagBOJ': 124, 'tagSA': 'math'}, {'tagBOJ': ..."
1096,종이 접기,,75,18,False,3.2133,"[{'tagBOJ': 125, 'tagSA': 'bruteforcing'}]"
1097,마법의 문자열,,362,16,False,2.0331,"[{'tagBOJ': 125, 'tagSA': 'bruteforcing'}, {'t..."


In [301]:
file_modified:str = f"{DATA_DIR_MODIFIED}/boj.problem.modified.{NUM_FROM}.{NUM_TO}.{date_str}.csv"
df_copy.to_csv(file_modified)

In [302]:
tags_raw = df["tags"].copy()

In [303]:
tags_raw.keys()

Index([1000, 1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010, 1011,
       1012, 1013, 1014, 1015, 1016, 1017, 1018, 1019, 1020, 1021, 1022, 1023,
       1024, 1025, 1026, 1027, 1028, 1029, 1030, 1031, 1032, 1033, 1034, 1035,
       1036, 1037, 1038, 1039, 1040, 1041, 1042, 1043, 1044, 1045, 1046, 1047,
       1048, 1049, 1050, 1051, 1052, 1053, 1054, 1055, 1056, 1057, 1058, 1059,
       1060, 1061, 1062, 1063, 1064, 1065, 1066, 1067, 1068, 1069, 1070, 1071,
       1072, 1073, 1074, 1075, 1076, 1077, 1078, 1079, 1080, 1081, 1082, 1083,
       1084, 1085, 1086, 1087, 1088, 1089, 1090, 1091, 1092, 1093, 1094, 1095,
       1096, 1097, 1098, 1099],
      dtype='int64', name='problemId')

In [323]:
tags_modified:list[list] = []

for id in list(tags_raw.keys()):
    for tag in tags_raw.loc[id]:
        tag_modified:list = [tag["bojTagId"], tag["key"]]
        tag_name_ko:str = ""
        tag_name_en:str = ""
        for dname in tag["displayNames"]:
            if dname["language"] == "ko":
                tag_name_ko = dname["name"]
            if dname["language"] == "en":
                tag_name_en = dname["name"]
        tag_modified.append(tag_name_ko)
        tag_modified.append(tag_name_en)
        tag_modified.append(tag["aliases"])
        if not tag_modified in tags_modified:
            tags_modified.append(tag_modified)
tags_modified = sorted(tags_modified, key=lambda _ : _[0])

In [325]:
df_tags = pd.DataFrame(tags_modified, columns=["tagBOJ", "tagSA", "nameKo", "nameEn", "aliases"])
df_tags.index = df_tags["tagBOJ"]
del df_tags["tagBOJ"]

In [327]:
file_modified:str = f"{DATA_DIR_MODIFIED}/boj.problem.tags.{NUM_FROM}.{NUM_TO}.{date_str}.csv"
df_tags.to_csv(file_modified)